# Tier 3 Assignment Set 2: Analysis and ML

---

These exercises cover:
- Promoter motif frequency analysis
- Multiple testing correction
- k-NN classification from scratch
- Variance and mutual information feature selection
- Random forest for cancer classification
- Pharmacogenomics variant interpretation

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import math
from collections import defaultdict
from scipy import stats

---

## Problem 1: Promoter Motif Frequency (1 star)

Given a set of promoter sequences (200 bp upstream of TSS), calculate the frequency
of three common promoter motifs and their position distributions:

- TATA box: `TATAAA`
- GC box:   `GGGCGG`
- CAAT box: `CCAAT`

**Expected output:**
```python
{
    'TATAAA': {'count': 8, 'frequency': 0.8, 'positions': [140, 145, ...]},
    ...
}
```
where `frequency` is the fraction of sequences that contain the motif at least once.

**Grading rubric:**
- Correctly counts all (possibly overlapping) occurrences per sequence (1 pt)
- Correctly reports per-motif position list across all sequences (1 pt)
- Correctly computes fraction of sequences containing each motif (1 pt)

In [ ]:
PROMOTER_MOTIFS = {
    'TATA_box': 'TATAAA',
    'GC_box':   'GGGCGG',
    'CAAT_box': 'CCAAT',
}


def analyze_promoter_motifs(sequences: list[str]) -> dict:
    """
    Analyze the frequency and position distribution of promoter motifs.

    Args:
        sequences: List of promoter sequences (each ~200 bp).

    Returns:
        Dict keyed by motif name, each value is a dict with:
        'count' (total occurrences), 'frequency' (fraction of seqs with >= 1 hit),
        'positions' (list of positions across all sequences).
    """
    # YOUR CODE HERE
    pass


# Test
np.random.seed(1)
bases = list('ACGT')

promoters = [''.join(np.random.choice(bases, 200)) for _ in range(10)]
# Inject known motifs
for i in range(8):
    pos = np.random.randint(130, 160)
    promoters[i] = promoters[i][:pos] + 'TATAAA' + promoters[i][pos + 6:]
for i in range(5):
    pos = np.random.randint(50, 100)
    promoters[i] = promoters[i][:pos] + 'CCAAT' + promoters[i][pos + 5:]

results = analyze_promoter_motifs(promoters)
for name, data in results.items():
    print(f"{name}: count={data['count']}, frequency={data['frequency']:.2f}, "
          f"positions_sample={data['positions'][:5]}")

---

## Problem 2: Multiple Testing Correction (2 stars)

Implement Bonferroni and Benjamini-Hochberg (BH) FDR correction from scratch.
Given a list of p-values, return adjusted p-values for both methods.
Report how many tests are significant at alpha = 0.05 under each method.

**Bonferroni:** `adjusted_p = min(p * n, 1.0)` where n = number of tests

**Benjamini-Hochberg:**
1. Sort p-values in ascending order, tracking original indices
2. Assign rank `k` (1-based)
3. Adjusted p at rank k: `min(p_k * n / k, 1.0)`
4. Enforce monotonicity from largest to smallest rank: `adj_p[k] = min(adj_p[k], adj_p[k+1])`

**Grading rubric:**
- Correct Bonferroni implementation (1 pt)
- Correct BH implementation including monotonicity step (2 pts)
- Results correctly returned in original p-value order (1 pt)

In [ ]:
def bonferroni_correction(pvalues: list[float]) -> list[float]:
    """
    Apply Bonferroni correction to a list of p-values.

    Args:
        pvalues: Raw p-values.

    Returns:
        Adjusted p-values in the same order as input.
    """
    # YOUR CODE HERE
    pass


def bh_correction(pvalues: list[float]) -> list[float]:
    """
    Apply Benjamini-Hochberg FDR correction to a list of p-values.

    Args:
        pvalues: Raw p-values.

    Returns:
        Adjusted p-values (q-values) in the same order as input.
    """
    # YOUR CODE HERE
    pass


# Test
np.random.seed(10)
# 90 null p-values + 10 truly significant ones
null_pvals = list(np.random.uniform(0.1, 1.0, 90))
sig_pvals  = list(np.random.uniform(0.0001, 0.005, 10))
pvals = null_pvals + sig_pvals

bonf = bonferroni_correction(pvals)
bh   = bh_correction(pvals)

alpha = 0.05
print(f"Total tests: {len(pvals)}")
print(f"Significant (raw p < {alpha}):         {sum(p < alpha for p in pvals)}")
print(f"Significant (Bonferroni adj < {alpha}): {sum(p < alpha for p in bonf)}")
print(f"Significant (BH adj < {alpha}):         {sum(p < alpha for p in bh)}")

---

## Problem 3: k-NN Classifier (2 stars)

Implement k-nearest neighbors from scratch (no sklearn). Given a gene expression
matrix with labeled samples (tumor vs normal), classify new samples using Euclidean
distance. Report accuracy using leave-one-out cross-validation (LOOCV).

**LOOCV:** For each sample i, train on all other samples and predict label for i.
Accuracy = fraction of correctly predicted labels.

**Grading rubric:**
- Correct Euclidean distance and majority-vote logic (2 pts)
- Correct leave-one-out cross-validation loop (1 pt)
- Reports final accuracy (1 pt)

In [ ]:
def knn_predict(
    X_train: np.ndarray,
    y_train: list,
    X_test: np.ndarray,
    k: int = 3,
) -> list:
    """
    Classify test samples using k-nearest neighbors.

    Args:
        X_train: Training feature matrix, shape (n_train, n_features).
        y_train: Training labels, length n_train.
        X_test: Test feature matrix, shape (n_test, n_features).
        k: Number of nearest neighbors to consider.

    Returns:
        List of predicted labels for each test sample.
    """
    # YOUR CODE HERE
    pass


def loocv_accuracy(
    X: np.ndarray,
    y: list,
    k: int = 3,
) -> float:
    """
    Estimate classification accuracy using leave-one-out cross-validation.

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y: Labels, length n_samples.
        k: Number of nearest neighbors.

    Returns:
        LOOCV accuracy (fraction of correct predictions).
    """
    # YOUR CODE HERE
    pass


# Test: synthetic expression data
np.random.seed(99)
n_genes = 20

# Tumor samples: higher expression of genes 0-9
tumor = np.random.randn(15, n_genes)
tumor[:, :10] += 2.0

# Normal samples: lower expression
normal = np.random.randn(15, n_genes)

X = np.vstack([tumor, normal])
y = ['tumor'] * 15 + ['normal'] * 15

acc = loocv_accuracy(X, y, k=3)
print(f"LOOCV accuracy (k=3): {acc:.3f}")

acc5 = loocv_accuracy(X, y, k=5)
print(f"LOOCV accuracy (k=5): {acc5:.3f}")

---

## Problem 4: Feature Selection (2 stars)

Given a gene expression matrix, implement two feature selection strategies:

1. **Variance-based**: rank genes by variance across all samples (descending)
2. **Mutual information**: rank genes by mutual information with the class label

Select the top 50 genes by each method and report the overlap (Jaccard index).

**Mutual information** (discrete approximation): bin expression values into quartiles,
then compute `MI(X; Y) = sum_x sum_y P(x,y) * log2(P(x,y) / (P(x)*P(y)))`.

**Grading rubric:**
- Correct variance ranking (1 pt)
- Correct mutual information calculation (2 pts)
- Correct Jaccard index computation and reported overlap (1 pt)

In [ ]:
def variance_selection(X: np.ndarray, top_n: int = 50) -> list[int]:
    """
    Select top_n genes by variance across samples.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        top_n: Number of genes to select.

    Returns:
        List of gene indices sorted by descending variance.
    """
    # YOUR CODE HERE
    pass


def mutual_information_selection(
    X: np.ndarray,
    y: list,
    top_n: int = 50,
    n_bins: int = 4,
) -> list[int]:
    """
    Select top_n genes by mutual information with class label.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        y: Class labels, length n_samples.
        top_n: Number of genes to select.
        n_bins: Number of quantile bins for discretizing expression values.

    Returns:
        List of gene indices sorted by descending mutual information.
    """
    # YOUR CODE HERE
    pass


def jaccard_index(set_a: list[int], set_b: list[int]) -> float:
    """
    Compute Jaccard similarity between two sets.

    Returns:
        |A ∩ B| / |A ∪ B|
    """
    # YOUR CODE HERE
    pass


# Test
np.random.seed(5)
n_samples, n_genes = 60, 200
y_labels = ['tumor'] * 30 + ['normal'] * 30

X_expr = np.random.randn(n_samples, n_genes)
# Make genes 0-49 informative: higher in tumors
X_expr[:30, :50] += 3.0

var_genes = variance_selection(X_expr, top_n=50)
mi_genes  = mutual_information_selection(X_expr, y_labels, top_n=50)

j = jaccard_index(var_genes, mi_genes)
overlap = set(var_genes) & set(mi_genes)
print(f"Variance top-50 sample: {sorted(var_genes)[:10]}")
print(f"MI top-50 sample:        {sorted(mi_genes)[:10]}")
print(f"Overlap size: {len(overlap)} genes")
print(f"Jaccard index: {j:.3f}")

---

## Problem 5: Random Forest for Cancer Classification (3 stars)

Build a random forest classifier to distinguish cancer types from gene expression data.
Use `sklearn`. Requirements:

1. Split data: 80% train / 20% test with `random_state=42`
2. Train a `RandomForestClassifier` with 100 trees
3. Report accuracy, precision, recall, and F1 score on the test set
4. Identify and print the top 20 most important genes (by feature importance)

**Grading rubric:**
- Correct train/test split and model training (1 pt)
- Correct metrics report (precision, recall, F1 per class) (1 pt)
- Correct extraction and reporting of top-20 feature importances (1 pt)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


def train_cancer_classifier(
    X: np.ndarray,
    y: list,
    gene_names: list[str],
    n_estimators: int = 100,
    random_state: int = 42,
) -> dict:
    """
    Train and evaluate a Random Forest cancer classifier.

    Args:
        X: Expression matrix, shape (n_samples, n_genes).
        y: Cancer type labels.
        gene_names: List of gene names, length n_genes.
        n_estimators: Number of trees in the forest.
        random_state: Random seed.

    Returns:
        Dict with keys:
            'model': fitted RandomForestClassifier
            'accuracy': float
            'report': classification report string
            'top_genes': list of (gene_name, importance) for top 20 genes
    """
    # YOUR CODE HERE
    pass


# Test: 3-class synthetic cancer data
np.random.seed(42)
n_samples_per_class = 50
n_genes = 500

gene_names = [f'GENE_{i:04d}' for i in range(n_genes)]

# Three cancer types with distinct expression signatures
lung   = np.random.randn(n_samples_per_class, n_genes)
breast = np.random.randn(n_samples_per_class, n_genes)
colon  = np.random.randn(n_samples_per_class, n_genes)

lung[:, :50]       += 3.0   # genes 0-49 up in lung
breast[:, 100:150] += 3.0   # genes 100-149 up in breast
colon[:, 200:250]  += 3.0   # genes 200-249 up in colon

X_cancer = np.vstack([lung, breast, colon])
y_cancer = ['lung'] * n_samples_per_class + ['breast'] * n_samples_per_class + \
           ['colon'] * n_samples_per_class

results = train_cancer_classifier(X_cancer, y_cancer, gene_names)
print(f"Test accuracy: {results['accuracy']:.3f}\n")
print(results['report'])
print("Top 20 genes:")
for gene, imp in results['top_genes']:
    print(f"  {gene}: {imp:.4f}")

---

## Problem 6: Pharmacogenomics Variant Interpretation (3 stars)

Given a list of patient variants (gene, variant, zygosity) and a drug-gene interaction
table, write a function that generates a pharmacogenomics report:
- Which drugs may have altered efficacy
- Recommended dosage adjustment
- Confidence level of each recommendation

**Input:**
```python
patient_variants = [
    {'gene': 'CYP2D6', 'variant': '*4/*4', 'zygosity': 'homozygous'},
    ...
]
drug_interactions = [
    {
        'gene': 'CYP2D6', 'variant': '*4',
        'drug': 'Codeine', 'effect': 'poor_metabolizer',
        'recommendation': 'avoid', 'confidence': 'high'
    },
    ...
]
```

**Expected output:**
```python
[
    {
        'drug': 'Codeine',
        'gene': 'CYP2D6',
        'effect': 'poor_metabolizer',
        'recommendation': 'avoid',
        'confidence': 'high',
        'zygosity': 'homozygous'
    },
    ...
]
```

**Grading rubric:**
- Correctly matches variants to drug interactions (checking that the variant allele
  appears in the patient's genotype string) (2 pts)
- Returns structured report entries with all required fields (1 pt)
- Deduplicates entries (same drug + gene should not appear twice) (1 pt)

In [ ]:
def generate_pgx_report(
    patient_variants: list[dict],
    drug_interactions: list[dict],
) -> list[dict]:
    """
    Generate a pharmacogenomics (PGx) clinical report for a patient.

    Matches patient variants to known drug-gene interactions.
    A drug interaction entry matches when:
      - The gene matches, AND
      - The interaction variant allele (e.g. '*4') appears in the patient's
        variant string (e.g. '*4/*4' or '*1/*4').

    Args:
        patient_variants: List of patient variant dicts with
            'gene', 'variant', 'zygosity'.
        drug_interactions: List of drug-gene interaction dicts with
            'gene', 'variant', 'drug', 'effect', 'recommendation', 'confidence'.

    Returns:
        Deduplicated list of report entries. Each entry contains:
        'drug', 'gene', 'effect', 'recommendation', 'confidence', 'zygosity'.
    """
    # YOUR CODE HERE
    pass


# Test
patient_variants = [
    {'gene': 'CYP2D6', 'variant': '*4/*4', 'zygosity': 'homozygous'},
    {'gene': 'CYP2C19', 'variant': '*2/*17', 'zygosity': 'compound_heterozygous'},
    {'gene': 'TPMT', 'variant': '*1/*1', 'zygosity': 'homozygous_reference'},
    {'gene': 'DPYD', 'variant': '*2A/*1', 'zygosity': 'heterozygous'},
]

drug_interactions = [
    {'gene': 'CYP2D6', 'variant': '*4', 'drug': 'Codeine',
     'effect': 'poor_metabolizer', 'recommendation': 'avoid', 'confidence': 'high'},
    {'gene': 'CYP2D6', 'variant': '*4', 'drug': 'Tamoxifen',
     'effect': 'reduced_efficacy', 'recommendation': 'consider_alternative',
     'confidence': 'high'},
    {'gene': 'CYP2C19', 'variant': '*2', 'drug': 'Clopidogrel',
     'effect': 'reduced_activation', 'recommendation': 'avoid', 'confidence': 'high'},
    {'gene': 'CYP2C19', 'variant': '*17', 'drug': 'Omeprazole',
     'effect': 'ultrarapid_metabolizer', 'recommendation': 'increase_dose',
     'confidence': 'moderate'},
    {'gene': 'DPYD', 'variant': '*2A', 'drug': 'Fluorouracil',
     'effect': 'reduced_metabolism', 'recommendation': 'reduce_dose_50pct',
     'confidence': 'high'},
    {'gene': 'TPMT', 'variant': '*3A', 'drug': 'Azathioprine',
     'effect': 'poor_metabolizer', 'recommendation': 'reduce_dose', 'confidence': 'high'},
]

report = generate_pgx_report(patient_variants, drug_interactions)
print(f"PGx report: {len(report)} actionable interactions found\n")
for entry in report:
    print(f"  Drug: {entry['drug']:<20} Gene: {entry['gene']:<10} "
          f"Effect: {entry['effect']:<25} Rec: {entry['recommendation']:<30} "
          f"Conf: {entry['confidence']}")

---

## Summary

In this assignment you worked with:

1. **Promoter Motif Analysis**: Frequency and positional distribution of regulatory elements
2. **Multiple Testing Correction**: Bonferroni and Benjamini-Hochberg FDR
3. **k-NN from Scratch**: Distance-based classification with LOOCV evaluation
4. **Feature Selection**: Variance and mutual information methods, Jaccard overlap
5. **Random Forest**: sklearn-based cancer classification with feature importances
6. **Pharmacogenomics**: Structured variant-drug interaction reporting

---